# 04 Asset Swaps

**Book:** *Fixed Income Relative Value Analysis (2nd ed.)*  
**Focus:** Chapter 12 asset swaps, swap-spread intuition, and pricing inputs.


## Goals

1. Load the local risk-free and funding proxies.
2. Scaffold the inputs needed for asset swap pricing.
3. Record what market data is still missing for a proper implementation.
4. Leave placeholder cells for swap-spread driver analysis.


In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

REPO_ROOT = Path("/Users/zelin/Desktop/PA Investment/Invest_strategy")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from alpha_research.quant_data.api import get_data

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

START = "2024-02-26"
END = "2026-02-27"
BOOK_PDF = Path(r"""/Users/zelin/Desktop/阅读学习/Fixed Income Relative Value Analysis + Website A Practitioner’s Guide to the Theory, Tools, and Trades 2nd.pdf""")

print("Expected environment: conda activate ibkr-analytics && export PYTHONPATH=.")
print("Book PDF exists:", BOOK_PDF.exists())
print("Repo root:", REPO_ROOT)


In [ ]:
FRED_DIR = REPO_ROOT / "data" / "market_data" / "fred"
PRICES_DIR = REPO_ROOT / "data" / "market_data" / "prices"

def _filter_date(frame: pd.DataFrame, start=START, end=END, date_col="date"):
    out = frame.copy()
    out[date_col] = pd.to_datetime(out[date_col])
    return out[(out[date_col] >= start) & (out[date_col] <= end)]

def load_fred_series(series_ids, start=START, end=END):
    frames = []
    for parquet_file in sorted(FRED_DIR.glob("*.parquet")):
        df = pd.read_parquet(parquet_file)
        if "series_id" not in df.columns:
            continue
        sub = df[df["series_id"].isin(series_ids)]
        if not sub.empty:
            frames.append(_filter_date(sub, start=start, end=end))

    if not frames:
        return pd.DataFrame()

    joined = pd.concat(frames, ignore_index=True).drop_duplicates(["date", "series_id"])
    wide = (
        joined.pivot(index="date", columns="series_id", values="value")
        .sort_index()
        .apply(pd.to_numeric, errors="coerce")
    )
    wide.index = pd.to_datetime(wide.index)
    return wide

def load_local_price_series(tickers, start=START, end=END, value_col="close"):
    frames = []
    for parquet_file in sorted(PRICES_DIR.glob("*.parquet")):
        df = pd.read_parquet(parquet_file)
        if "ticker" not in df.columns or value_col not in df.columns:
            continue
        sub = df[df["ticker"].isin(tickers)]
        if not sub.empty:
            frames.append(_filter_date(sub, start=start, end=end))

    if not frames:
        return pd.DataFrame()

    joined = pd.concat(frames, ignore_index=True).drop_duplicates(["date", "ticker"])
    wide = (
        joined.pivot(index="date", columns="ticker", values=value_col)
        .sort_index()
        .apply(pd.to_numeric, errors="coerce")
    )
    wide.index = pd.to_datetime(wide.index)
    return wide


In [ ]:
base_rates = load_fred_series(["SOFR", "DFEDTARU", "DGS2", "DGS5", "DGS10", "DGS30"]).dropna()
base_rates.tail()


In [ ]:
base_rates[["SOFR", "DFEDTARU"]].plot(title="Funding and Policy Proxies")
plt.show()


## Minimal pricing schema scaffold

A proper asset swap implementation needs:

- bond clean or dirty price
- coupon schedule and accrual conventions
- swap curve / discount curve
- funding spread assumptions
- package cashflow conventions

The local lake does not yet contain the full market data required, so this notebook provides structure and placeholders.


In [ ]:
example_bond = {
    "issuer": "UST proxy",
    "maturity_years": 5.0,
    "coupon": 0.04,
    "clean_price": np.nan,  # TODO: replace with actual bond price input
    "payment_frequency": 2,
}

example_swap_inputs = {
    "floating_reference": "SOFR",
    "spread_guess_bp": 0.0,
    "discount_curve_proxy": "Treasury fitted curve / swap curve placeholder",
}

example_bond, example_swap_inputs


In [ ]:
# TODO: implement bond cashflow schedule generation.
# TODO: implement discount-factor curve input.
# TODO: solve for the asset swap spread that prices the package to par.


## Spread-driver placeholders

The book highlights several swap-spread drivers. Add diagnostics here for:

- policy / funding regime
- Treasury richness / cheapness
- collateral / capital proxy variables
- cyclicality against macro indicators and stress proxies


In [ ]:
driver_panel = load_fred_series(["SOFR", "DFEDTARU", "T10Y2Y", "T10Y3M", "T10YIE", "T5YIE"]).dropna()
driver_panel.tail()


In [ ]:
# TODO: add proxy regressions once a usable asset swap spread time series is available.
# Candidate future data additions:
# - SOFR asset swap spreads by tenor
# - government/corporate cash bond prices
# - repo / collateral specialness
